In [1]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [3]:
import json
import logging
import torch
from dataclasses import dataclass
import wandb
from dotenv import load_dotenv
import evaluate
from datasets import Dataset, ClassLabel, DatasetDict, load_dataset
from transformers import (
    pipeline,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import numpy as np


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

In [4]:
@dataclass
class TrainingConfig:
    model_name: str = "microsoft/deberta-v3-small"
    setfit_model: str = "sentence-transformers/paraphrase-mpnet-base-v2"
    output_dir: str = "model_finetuned/"
    epochs: int = 5
    batch_size: int = 16
    lr: float = 2e-5
    max_length: int = 256
    test_size: float = 0.2
    seed: int = 42
    few_shot_n: int = 8     
    zeroshot_batch: int = 32 
    wandb_project: str = "ticket-triage"
 

cfg = TrainingConfig()

CATEGORIES   = ["billing", "technical", "shipping", "account", "general"]

ds = Dataset.from_json(cfg.dataset_path)

id2category = {i: label for i, label in enumerate(CATEGORIES)}
category2id = {label: i for i, label in enumerate(CATEGORIES)}

f1_metric = evaluate.load('f1')
acc_metric = evaluate.load('accuracy')

AttributeError: 'TrainingConfig' object has no attribute 'dataset_path'

In [71]:
from collections import Counter

Counter(ds['category'])

Counter({'technical': 17655,
         'account': 5639,
         'biling': 3019,
         'general': 1875,
         'shipping': 1470})

In [5]:
from datasets import load_dataset
ds = load_dataset('kentokamg/ticket-dataset')
label_names = sorted(set(ds['category']))
label_feature = ClassLabel(names=label_names)
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

ds = ds.cast_column('category', label_feature)
ds = ds.train_test_split(test_size=cfg.test_size, stratify_by_column='category',seed=cfg.seed)

Generating train split: 100%|██████████| 29658/29658 [00:00<00:00, 171740.41 examples/s]


KeyError: 'category'

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'category', 'urgency', 'team'],
        num_rows: 29658
    })
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_dataset(hf_dataset):
    def tokenize_function(batch):
        tokens = tokenizer(
            batch['text'],
            truncation=True,
            max_length=cfg.max_length
        )
        tokens['labels'] = batch['category']
        return tokens
    return hf_dataset.map(tokenize_function, batched=True, remove_columns=['category', 'urgency', 'team'])

Map: 100%|██████████| 5932/5932 [00:00<00:00, 23439.77 examples/s]


In [86]:
ds['train']['text'][0]

'Dear Customer Support Team,I am reaching out to address the urgent requirement to strengthen security protocols and compliance procedures related to EMR/PACS integrations, telehealth systems, and Internet of Medical Things (IoMT) devices. In the healthcare industry, ensuring strong security measures is crucial to safeguard confidential patient data and guarantee seamless, protected service provision.Our existing infrastructure demands ongoing attention through continuous DevSecOps practices to quickly identify and remedy vulnerabilities. Adopting a proactive strategy will help us maintain high security standards.'

In [ ]:
def zero_shot(ds: DatasetDict) -> dict:
    logger.info(f'Evaluando zero-shot con {cfg.model_name}')
    classifier = pipeline('zero-shot-classification', 
                          model=cfg.model_name,
                          device=0, 
                          batch_size=cfg.zeroshot_batch)
    test_texts = ds['test']['text']
    test_labels = ds['test']['category']
    results = classifier(
        test_texts,
        candidate_labels=CATEGORIES,
        multi_label = False
    )
    predictions = [CATEGORIES.index(r['labels'][0]) for r in results]
    f1 = f1_metric.compute(predictions=predictions, references=test_labels, average='macro')
    accuracy = acc_metric.compute(predictions=predictions, references=test_labels)

    wandb.log({
        'zero-shot/f1_macro': f1['f1'],
        'zero-shot/accuracy': accuracy['accuracy']
    })
    logger.info(f"Zero-shot — F1: {f1['f1']:.3f} | Acc: {accuracy['accuracy']:.3f}")
    return {"f1": f1["f1"], "acc": accuracy["accuracy"]}
    